# Notebook principal — INF6243

Ce notebook exécute le pipeline complet et affiche les résultats clés.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown, Image

import sys
sys.path.insert(0, "Code")
from notebook_workflow import (
    build_models_table,
    build_runs_comparison_table,
    load_report,
    run_all_configs,
)
from result_interpreter import interpret_report

In [ ]:
# Strategie multi-runs: configurations definies dans le notebook.
def get_default_runs() -> dict:
    """Retourne les configurations de runs par défaut.

    Paramètres de configuration (appliqués dans chaque run):
        - max_samples (int | None): taille max de l'échantillon stratifié.
            Valeurs recommandées: None (100% du dataset) ou entier > 1000.
        - distilbert_epochs (int): nombre d'epochs pour DistilBERT.
            Range recommandé: 1 à 5 (1 rapide, 2-3 standard, >3 plus coûteux).
        - include_distilbert (bool): inclut ou non DistilBERT dans la comparaison.
            Valeurs: True ou False.
        - algorithm_switches (dict[str, bool]): activation fine de chaque algorithme.
            Ex: {"AdaBoost": False, "DistilBERT": True}. Si absent, tous les modèles sont actifs
            (avec `include_distilbert` comme compatibilité pour DistilBERT).
        - test_size (float): proportion du jeu de test.
            Range valide: 0.1 à 0.4 (strictement entre 0 et 1).
        - val_size (float): proportion du jeu de validation.
            Range valide: 0.05 à 0.3 (strictement entre 0 et 1).
        - cv_folds (int): nombre de folds pour la validation croisée.
            Range recommandé: 3 à 10.
        - scoring (str): métrique sklearn utilisée pour GridSearchCV.
            Valeurs typiques: "f1_macro", "accuracy", "precision_macro", "recall_macro".
        - model_param_overrides (dict[str, dict]): paramètres fixes par modèle.
            Exemple utile: `{"DistilBERT": {"epochs": 2, "batch_size": 8, "max_length": 160}}`.
            Impact: permet d'ajuster précisément le coût mémoire/temps et la capacité d'adaptation.
        - model_grid_overrides (dict[str, dict[str, list]]): surcharge des grilles GridSearch par modèle.
            Exemple utile: `{"MLPClassifier": {"clf__alpha": [1e-4, 1e-3]}}`.
            Impact: plus de combinaisons = meilleure chance d'optimum, mais coût CPU/RAM plus élevé.
        - selection_weights (tuple[float, float, float, float]): poids
            `(validation, test, cv, hate_recall)` du score de sélection final.
            Range recommandé: chaque poids entre 0 et 1, somme = 1.0.
        - hate_recall_floor (float): seuil minimum de recall pour `hate_speech` sur test.
            Range recommandé: 0.30 à 0.50.
        - hate_recall_penalty (float): pénalité soustraite au score si le seuil n'est pas atteint.
            Range recommandé: 0.01 à 0.05.
        - random_state (int): seed de reproductibilité.
            Valeur recommandée: entier positif (ex: 42).
        - DISTILBERT_PROXY_PENALTY (float): malus de prudence appliqué aux runs
            incluant DistilBERT quand `cv_fallback_for_models` contient DistilBERT.
            Range recommandé: 0.00 à 0.05.

    Notes:
        Le score de sélection est calculé par:
        `selection_score = w_val * val_f1_macro + w_test * test_f1_macro + w_cv * cv_f1_macro_mean + w_hate * hate_recall_test`.
        Si `hate_recall_test < hate_recall_floor`, une pénalité est soustraite.
        Plus `w_cv` est élevé, plus la stabilité inter-fold est valorisée.

    Retour:
        Dictionnaire des runs prêts à exécuter par `run_all_configs()`.
    """
    # Parametres communs (non mouvants entre les runs)
    TEST_SIZE = 0.2
    VAL_SIZE = 0.1
    SCORING = "f1_macro"
    RANDOM_STATE = 42

    # Overriding centralisé des hyperparamètres.
    # Utiliser `_BASE` si vous voulez garder les réglages model_zoo par défaut.
    MODEL_PARAM_OVERRIDES_BASE = {}
    MODEL_GRID_OVERRIDES_BASE = {}

    # DistilBERT: profils conseillés pour contrôler coût vs qualité.
    # - epochs: 1-3 pour itérations, 4-5 pour consolidation finale.
    # - batch_size: 8 si VRAM limitée, 16 compromis, 32 si GPU confortable.
    # - max_length: 128 défaut, 160/192 pour plus de contexte (coût en hausse).
    DISTILBERT_PARAMS_EP2 = {
        "DistilBERT": {
            "epochs": 2,
            "batch_size": 16,
            "max_length": 160,
            "learning_rate": 3e-5,
            "weight_decay": 0.01,
        }
    }
    DISTILBERT_PARAMS_EP3 = {
        "DistilBERT": {
            "epochs": 3,
            "batch_size": 16,
            "max_length": 160,
            "learning_rate": 3e-5,
            "weight_decay": 0.01,
        }
    }

    # MLP: exemple de grille étendue (plus coûteuse) pour exploration dédiée.
    MLP_GRID_WIDE = {
        "MLPClassifier": {
            "svd__n_components": [20, 40, 60],
            "clf__hidden_layer_sizes": [(128,), (256, 128), (256, 256)],
            "clf__alpha": [1e-4, 5e-4, 1e-3],
            "clf__learning_rate_init": [1e-3, 5e-4, 3e-4],
        }
    }

    # Parametres partages par sous-groupes de runs
    CV_FOLDS_STANDARD = 5
    CV_FOLDS_FAST = 3

    # Weights recommandés pour selection_score = val + test + cv + hate_recall
    # validation: garde-fou tuning, test: performance réelle, cv: stabilité, hate_recall: protection classe minoritaire.
    WEIGHTS_BALANCED = (0.30, 0.35, 0.20, 0.15)
    WEIGHTS_TEST_PRIORITY = (0.25, 0.45, 0.15, 0.15)

    # Weights additionnels pour scenarios méthodologiques
    WEIGHTS_METHOD_STRICT = (0.45, 0.20, 0.20, 0.15)
    WEIGHTS_CV_HEAVY = (0.25, 0.25, 0.35, 0.15)
    WEIGHTS_DISTILBERT_SAFE = (0.30, 0.40, 0.15, 0.15)

    # Garde-fou classe minoritaire hate_speech
    HATE_RECALL_FLOOR = 0.40
    HATE_RECALL_PENALTY = 0.03

    # Activation fine des algorithmes (paramètre run-level)
    ALGORITHM_SWITCHES_ALL = {
        "NaiveBayes": True,
        "LogisticRegression": True,
        "LinearSVC": True,
        "KNN": True,
        "DecisionTree": True,
        "RandomForest": True,
        "AdaBoost": True,
        "MLPClassifier": True,
        "DistilBERT": True,
    }
    ALGORITHM_SWITCHES_CLASSIC_ONLY = {
        **ALGORITHM_SWITCHES_ALL,
        "DistilBERT": False,
    }

    return {
        # Baselines
        "run_a_data_balance": {
            "why": (
                "Run A = baseline robuste sur plus de donnees. "
                "On utilise 100% des echantillons pour mieux couvrir la classe minoritaire hate_speech."
            ),
            "config": {
                "max_samples": None,
                "distilbert_epochs": 1,
                "include_distilbert": True,
                "test_size": TEST_SIZE,
                "val_size": VAL_SIZE,
                "cv_folds": CV_FOLDS_STANDARD,
                "scoring": SCORING,
                "model_param_overrides": MODEL_PARAM_OVERRIDES_BASE,
                "model_grid_overrides": MODEL_GRID_OVERRIDES_BASE,
                "selection_weights": WEIGHTS_BALANCED,
                "algorithm_switches": ALGORITHM_SWITCHES_ALL,
                "hate_recall_floor": HATE_RECALL_FLOOR,
                "hate_recall_penalty": HATE_RECALL_PENALTY,
                "random_state": RANDOM_STATE,
            },
        },
        # "run_b_data_low_balance": {
        #     "why": (
        #         "Run B = baseline robuste sur peu de donnees. "
        #         "On utilise 12000 max_samples pour évaluer l'effet d'échantillonnage."
        #     ),
        #     "config": {
        #         "max_samples": 12000,
        #         "distilbert_epochs": 1,
        #         "include_distilbert": True,
        #         "test_size": TEST_SIZE,
        #         "val_size": VAL_SIZE,
        #         "cv_folds": CV_FOLDS_STANDARD,
        #         "scoring": SCORING,
        #         "selection_weights": WEIGHTS_BALANCED,
        #         "algorithm_switches": ALGORITHM_SWITCHES_ALL,
        #         "random_state": RANDOM_STATE,
        #     },
        # },

        # Ligue classique (comparaison stricte, sans fallback DistilBERT)
        # "run_c_classic_focus": {
        #     "why": (
        #         "Run C = focus modeles classiques. "
        #         "On retire DistilBERT pour mesurer la performance pure TF-IDF+ML avec CV complete."
        #     ),
        #     "config": {
        #         "max_samples": None,
        #         "distilbert_epochs": 1,
        #         "include_distilbert": False,
        #         "test_size": TEST_SIZE,
        #         "val_size": VAL_SIZE,
        #         "cv_folds": CV_FOLDS_STANDARD,
        #         "scoring": SCORING,
        #         "selection_weights": WEIGHTS_TEST_PRIORITY,
        #         "algorithm_switches": ALGORITHM_SWITCHES_CLASSIC_ONLY,
        #         "hate_recall_floor": HATE_RECALL_FLOOR,
        #         "hate_recall_penalty": HATE_RECALL_PENALTY,
        #         "random_state": RANDOM_STATE,
        #     },
        # },
        "run_e_method_strict_classic": {
            "why": "Run E = sélection méthodologique stricte, test faiblement pondéré, classiques uniquement.",
            "config": {
                "max_samples": None,
                "distilbert_epochs": 1,
                "include_distilbert": False,
                "test_size": TEST_SIZE,
                "val_size": VAL_SIZE,
                "cv_folds": CV_FOLDS_STANDARD,
                "scoring": SCORING,
                "model_param_overrides": MODEL_PARAM_OVERRIDES_BASE,
                "model_grid_overrides": MODEL_GRID_OVERRIDES_BASE,
                "selection_weights": WEIGHTS_METHOD_STRICT,
                "algorithm_switches": ALGORITHM_SWITCHES_CLASSIC_ONLY,
                "hate_recall_floor": HATE_RECALL_FLOOR,
                "hate_recall_penalty": HATE_RECALL_PENALTY,
                "random_state": RANDOM_STATE,
            },
        },
        "run_f_cv_heavy_classic": {
            "why": "Run F = robustesse prioritaire via CV fort, classiques uniquement.",
            "config": {
                "max_samples": None,
                "distilbert_epochs": 1,
                "include_distilbert": False,
                "test_size": TEST_SIZE,
                "val_size": VAL_SIZE,
                "cv_folds": CV_FOLDS_STANDARD,
                "scoring": SCORING,
                "model_param_overrides": MODEL_PARAM_OVERRIDES_BASE,
                # Run F: on élargit la grille MLP pour explorer davantage en mode classique.
                "model_grid_overrides": MLP_GRID_WIDE,
                "selection_weights": WEIGHTS_CV_HEAVY,
                "algorithm_switches": ALGORITHM_SWITCHES_CLASSIC_ONLY,
                "hate_recall_floor": HATE_RECALL_FLOOR,
                "hate_recall_penalty": HATE_RECALL_PENALTY,
                "random_state": RANDOM_STATE,
            },
        },
        # "run_i_adaboost_heavy_classic": {
        #     "why": (
        #         "Run I = stress-test AdaBoost en contexte classique. "
        #         "Grille de recherche coûteuse sur 100% des données, CV standard, sans DistilBERT."
        #     ),
        #     "config": {
        #         "max_samples": None,
        #         "distilbert_epochs": 1,
        #         "include_distilbert": False,
        #         "test_size": TEST_SIZE,
        #         "val_size": VAL_SIZE,
        #         "cv_folds": CV_FOLDS_STANDARD,
        #         "scoring": SCORING,
        #         "selection_weights": WEIGHTS_TEST_PRIORITY,
        #         "algorithm_switches": {
        #             **ALGORITHM_SWITCHES_CLASSIC_ONLY,
        #             "AdaBoost": True,
        #         },
        #         "random_state": RANDOM_STATE,
        #     },
        # },

        # Ligue étendue avec DistilBERT (weights adaptés au fallback CV proxy)
        # "run_d_distilbert_ep2": {
        #     "why": (
        #         "Run D = focus deep learning (2 epochs). "
        #         "Pondération CV réduite pour limiter l'impact du fallback DistilBERT."
        #     ),
        #     "config": {
        #         "max_samples": None,
        #         "distilbert_epochs": 2,
        #         "include_distilbert": True,
        #         "test_size": TEST_SIZE,
        #         "val_size": VAL_SIZE,
        #         "cv_folds": CV_FOLDS_STANDARD,
        #         "scoring": SCORING,
        #         "selection_weights": WEIGHTS_DISTILBERT_SAFE,
        #         "algorithm_switches": ALGORITHM_SWITCHES_ALL,
        #         "hate_recall_floor": HATE_RECALL_FLOOR,
        #         "hate_recall_penalty": HATE_RECALL_PENALTY,
        #         "random_state": RANDOM_STATE,
        #     },
        # },
        "run_g_distilbert_safe_ep3": {
            "why": "Run G = DistilBERT (3 epochs) avec même compensation de pondération CV.",
            "config": {
                "max_samples": None,
                "distilbert_epochs": 3,
                "include_distilbert": True,
                "test_size": TEST_SIZE,
                "val_size": VAL_SIZE,
                "cv_folds": CV_FOLDS_STANDARD,
                "scoring": SCORING,
                # Run G: profil DistilBERT explicite pour pilotage reproductible.
                "model_param_overrides": DISTILBERT_PARAMS_EP3,
                "model_grid_overrides": MODEL_GRID_OVERRIDES_BASE,
                "selection_weights": WEIGHTS_DISTILBERT_SAFE,
                "algorithm_switches": ALGORITHM_SWITCHES_ALL,
                "hate_recall_floor": HATE_RECALL_FLOOR,
                "hate_recall_penalty": HATE_RECALL_PENALTY,
                "random_state": RANDOM_STATE,
            },
        },
    }


RUNS = get_default_runs()

# Regle paramétrable de compensation DistilBERT (CV proxy).
# Valeurs utiles: 0.00 (sans malus) a 0.05 (malus prudent fort).
DISTILBERT_PROXY_PENALTY = 0.02

workflow = run_all_configs(RUNS, distilbert_proxy_penalty=DISTILBERT_PROXY_PENALTY)
run_summary_df = workflow["run_summary_df"]
BEST_RUN = workflow["best_run"]
artifacts = workflow["artifacts"]
FIGURE_NAMES = workflow["figure_names"]

# Affichage clair: score brut vs score ajusté + indicateurs DistilBERT proxy
display(
    run_summary_df[
        [
            "run",
            "best_model",
            "include_distilbert",
            "distilbert_cv_proxy",
            "fairness_penalty",
            "best_selection_score",
            "adjusted_selection_score",
            "best_test_f1_macro",
        ]
    ]
)
print(f"Run de reference selectionne pour la suite (score ajusté): {BEST_RUN}")
artifacts

## Pourquoi separer en plusieurs runs ?

- **Ligue classique**: runs sans DistilBERT pour comparaison stricte avec CV complet.
- **Run AdaBoost heavy**: stress-test dédié pour mesurer le gain réel d'un boosting coûteux dans le cadre TF-IDF classique.
- **Ligue étendue**: runs avec DistilBERT, en réduisant le poids CV pour compenser le fallback `cv_mean = val_f1`.
- **Compensation réaliste**: un léger malus de prudence est appliqué au score de sélection des runs DistilBERT utilisant le CV proxy, afin d'éviter de surclasser artificiellement ces runs.

Cette séparation permet de comparer proprement les modèles classiques et deep learning tout en gardant une règle de décision transparente.

In [ ]:
report = load_report(Path(artifacts["report_path"]))

print("Meilleur modèle:", report.get("best_model"))
print("Métriques test:", report.get("best_model_test_metrics"))
print("DistilBERT:", report.get("distilbert_note"))
print("Configuration:", report.get("run_config"))
display(Markdown("**Justification:** " + report.get("best_model_selection_explanation", "")))

In [ ]:
# Tableau complet: tous les modèles, statut, métriques, hyperparamètres
build_models_table(report)

In [ ]:
# Interpreteur de resultats (script dedie)
interpreter_summary = interpret_report(report, weak_f1_threshold=0.55)
interpreter_summary

## Pourquoi `best_cv_score` peut être vide pour DistilBERT ?

- Les modèles classiques utilisent `GridSearchCV`, donc un `best_cv_score` est produit.
- DistilBERT est entraîné en fine-tuning direct (pas de GridSearchCV complet pour limiter le coût).
- Le fallback de robustesse est documenté dans `model_selection_method.cv_fallback_for_models`.

In [ ]:
# Comparaison inter-runs (avec compensation DistilBERT CV proxy)
reports_dir = Path(artifacts["report_path"]).parent
comparison_df = build_runs_comparison_table(
    reports_dir,
    distilbert_proxy_penalty=DISTILBERT_PROXY_PENALTY,
)

if comparison_df.empty:
    print("Aucun report multi-run detecte.")
else:
    ordered_cols = [
        "run",
        "best_model",
        "include_distilbert",
        "distilbert_cv_proxy",
        "fairness_penalty",
        "best_selection_score",
        "adjusted_selection_score",
        "best_test_f1_macro",
        "distilbert_note",
    ]
    display(comparison_df[ordered_cols])

    print("\n=== Ligue classique (sans DistilBERT) ===")
    classic_df = comparison_df[comparison_df["include_distilbert"] == False]
    if classic_df.empty:
        print("Aucun run classique trouvé.")
    else:
        display(classic_df[ordered_cols])

    print("\n=== Ligue étendue (avec DistilBERT) ===")
    distil_df = comparison_df[comparison_df["include_distilbert"] == True]
    if distil_df.empty:
        print("Aucun run DistilBERT trouvé.")
    else:
        display(distil_df[ordered_cols])

    print("Astuce: la comparaison est aussi disponible en figure (runs_comparison_overview.png).")

In [ ]:
fig_dir = Path(artifacts["figures_dir"])
for fig_name in FIGURE_NAMES:
    fig_path = fig_dir / fig_name
    if fig_path.exists():
        display(Markdown(f"### {fig_name}"))
        display(Image(filename=str(fig_path)))
    else:
        print(f"Figure absente: {fig_path}")